In [8]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from transformers import CLIPModel, CLIPProcessor
from tqdm import tqdm
import json
import faiss

In [9]:
def extract_clip_embeddings(image_folder: str = 'posters',clip_model: str = 'openai/clip-vit-base-patch32',batch_size: int = 32,device: str = None) -> Tuple[np.ndarray, List[Dict]]:
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model = CLIPModel.from_pretrained(clip_model).to(device)
    processor = CLIPProcessor.from_pretrained(clip_model)
    model.eval()
    
    img_folder = Path(image_folder)
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        image_files.extend(list(img_folder.glob(ext)))
    
    metadata = []
    for img_path in image_files:
        metadata.append({
            'image_path': str(img_path),
            'title': img_path.stem.replace('_', ' ')
        })
    
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(metadata), batch_size)):
            batch_data = metadata[i:i+batch_size]
            images = [Image.open(item['image_path']).convert('RGB') for item in batch_data]
            
            inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
            image_features = model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
            all_embeddings.append(image_features.cpu().numpy())
    
    embeddings = np.vstack(all_embeddings)
    return embeddings, metadata

In [10]:
def save_to_vector_db(
    embeddings: np.ndarray,
    metadata: List[Dict],
    db_path: str = './vector_db',
    collection_name: str = 'movie_posters'
) -> Dict:
    db_info = {
        'db_type': 'faiss',
        'db_path': db_path,
        'collection_name': collection_name
    }
    Path(db_path).mkdir(parents=True, exist_ok=True)
    embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings_normalized.astype('float32'))
    index_path = Path(db_path) / f'{collection_name}.index'
    faiss.write_index(index, str(index_path))
    db_info['index_path'] = str(index_path)
    with open(Path(db_path) / 'metadata.json', 'w') as f:
        json.dump(metadata, f)
    with open(Path(db_path) / 'db_info.json', 'w') as f:
        json.dump(db_info, f)
    return db_info


In [11]:
def find_similar_movies(
    query: str or int,
    db_path: str = './vector_db',
    top_k: int = 10
) -> List[Dict]:
    with open(Path(db_path) / 'db_info.json', 'r') as f:
        db_info = json.load(f)
    with open(Path(db_path) / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    if isinstance(query, str):
        query_idx = next((i for i, m in enumerate(metadata) if query.lower() in m['title'].lower()), None)
        if query_idx is None:
            raise ValueError(f"Movie '{query}' not found")
    else:
        query_idx = query
    index = faiss.read_index(db_info['index_path'])
    embeddings = np.zeros((index.ntotal, index.d), dtype='float32')
    index.reconstruct_n(0, index.ntotal, embeddings)
    query_normalized = embeddings[query_idx:query_idx+1]
    distances, indices = index.search(query_normalized, top_k + 1)
    similar = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx != query_idx:
            similar.append({
                'index': int(idx),
                'title': metadata[idx]['title'],
                'similarity': float(dist),
                'image_path': metadata[idx]['image_path']
            })
    return similar[:top_k]


In [12]:
def get_movie_image(movie_id: str or int, db_path: str = './vector_db') -> Image.Image:
    with open(Path(db_path) / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    if isinstance(movie_id, str):
        movie = next((m for m in metadata if movie_id.lower() in m['title'].lower()), None)
        if movie is None:
            raise ValueError(f"Movie '{movie_id}' not found")
    else:
        movie = metadata[movie_id]
    return Image.open(movie['image_path'])
